# Globular Cluster Membership — NGC 6397

*Archived project. Not currently on the schedule — see `homework/extras/README.md`.*

This combines two pieces that used to be separate: the guided M4 isochrone exercise
(formerly HW7) and the NGC 6397 classification project (formerly the final project).
They belong together — the first teaches the technique, the second applies it.

**Part 0** works through isochrones on M4, where the cluster membership is already known.
**Parts 1–2** turn the method loose on NGC 6397: train a classifier to find members, then
use an isochrone — physics the model never saw — to judge whether the candidates it finds
are real.

Some of this needs data from the repository's top-level `data/` directory
(`NGC6121-1.dat`, `m4_gaia_source.csv.gz`); the NGC 6397 files and the MIST isochrones are
in this project's own `data/`.


---
# Part 0 — Isochrones, using M4

A warm-up on a cluster whose membership is already established, before applying the same
idea to NGC 6397 in Part 2.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Isochrones

Recall that stellar clusters are gravitationally bound groupings of stars born at the same time.  By looking at the H-R diagram of a cluster -- specifically where the population _leaves_ the main sequence -- we can determine the age of the cluster.  To do this we must know the typical main sequence lifetime of stars, which depends on their mass, metalicity, etc. and for that we must turn to stellar modeling.  [MESA](https://docs.mesastar.org) is a stellar modeling library which incorporates the physics important for stellar evolution into 1-D models of stars.  The results useful for our purposes are stellar evolution tracks, the expected trajectory of the star through the H-R diagram as it evolves in time.

[MIST](http://waps.cfa.harvard.edu/MIST/index.html) is an effort to supply a large database of MESA stellar track simulations for the purpose of producing isochrones.  If you imagine stacking up a cluster's worth of stellar evolution tracks, slicing through them at fixed age will produce an isochrone.

I've used the [MIST isochrone interpolator](http://waps.cfa.harvard.edu/MIST/interp_isos.html) to produce several isochrones based on known properties of M4 (metalicity, reddening due to dust, etc.), which I've included in the `data/` directory.

There is a lot of info in these files.  To save ourselves a _big_ headache in parsing them, we'll make use of a python script authored by [Jieun Choi](https://github.com/jieunchoi).

In [ ]:
!wget -nc https://github.com/jieunchoi/MIST_codes/raw/master/scripts/read_mist_models.py

Now let's read in the file.

In [ ]:
import read_mist_models

filename = '../data/m4_isochrones.iso.cmd'
iso = read_mist_models.ISOCMD(filename)

The information we're after is in `iso.isocmds` (CMD: color-magnitude diagram).

In [ ]:
type(iso.isocmds), len(iso.isocmds)

We see this is a list with 7 elements.  These 7 CMDs are isochrones for 7 different ages, which we can see in the table of each.  Let's inspect the first one.

In [ ]:
iso_array = iso.isocmds[0]
type(iso_array)

This are numpy record arrays (basically numpy arrays with names columns).

In [ ]:
iso_array.dtype.names

For a more detailed description see [here](http://waps.cfa.harvard.edu/MIST/README_tables.pdf).  Most of these columns are synthetic photometry (magnitude measurements) for various telescopes and filters (we'll be using the `Gaia_X_EDR3` ones).  `log10_isochrone_age_yr` is what it sounds like, and is different for each of the 7 CMDs in the list.

In [ ]:
# Compute our color quantity (BP - RP) for the isochrone and plot G vs BP-RP for the isochrone. Don't forget to invert the y-axis!

This includes _all_ phases of stellar evolution.  If our focus is to date the cluster, we really only care about the main sequence and evolutionary phases immediately after (red giant) to find the turnoff.  `phase` indicates this the stellar evolution phase along the isochrone.  We can select the phases we're after:

In [ ]:
phase_sel = (iso_array['phase'] >= 0) & (iso_array['phase'] < 3)

In [ ]:
# Use the selection array to downselect the isochrone and plot it

One final tweak we must make is to account for the distance to the cluster.  Our y-axis isn't actually luminosity, it's the _apparent_ magnitude in the G-band.  What we've been calling `mg` in our previous use of Gaia data was actually derived from the G-band observations and corrected for the distance to each sourse based on parallax measurement (which gave us something like an intrinsic, rather than apparent, brightness).  We're going to know use the true G-band measurement, which is `phot_g_mean_mag`.

The isochrone simulation doesn't account for the faintness we would expect, so we'll need to do it ourselves by applying a [distance modulus](https://en.wikipedia.org/wiki/Distance_modulus).

The cluster is 2.2 kpc away.  We'll use some convenience functions from astropy to compute the corresponding distance modulus.

In [ ]:
!pip install --quiet astropy

In [ ]:
import astropy.coordinates as coord
import astropy.units as u

distance = 2.2 * u.kpc
distmod = coord.Distance(distance).distmod.value
distmod

In [ ]:
# Add this distance modulus to the G-band magnitude of the isochrone

# Overplot Gaia Observation

Finally, let's overplot the Gaia observations of confidently identified members of the M4 cluster.  Read in the cluster catalog we've worked with previously (`NGC6121-1.dat`), Gaia objects in the M4-neigborhood (`m4_gaia_source.csv`), and cross-match the catalogs to find the Gaia observations corresponding to the identified cluster members.

In [ ]:
# Read M4 (NGC6121) and Gaia catalogd and crossmatch them

Now plot the M4 cluster members with the isochrone!

In [ ]:
# plot M4 members and isochrone

How do they compare?  Explore the other isochrones.  Based on the fits (comparing by eye is sufficient), how old do you think 

---
*End of the M4 warm-up. Everything below is the NGC 6397 project.*

---


# Globular Cluster Member Identification: NGC 6397
NGC 6397 is the second closest globular cluster to Earth (second to M4).  Similar to our exploration of M4 using Gaia, we're going to make use of a catalog of confidently identified stellar members of the cluster to train classifiers capable of making such classifications from Gaia observations.

First we'll download the necessary data.  The catalog was originally pulled from [here](http://cdsarc.u-strasbg.fr/ftp/J/A+A/616/A12/), and can be found in `data/NGC6397-1.dat`.

From the Gaia archive we'll pull all objects in a 2 deg x 1.5 deg box centered on the cluster.  The query used is below, which was used to generate `data/gaia-NGC6397-neighborhood.csv`.

```sql
SELECT TOP 500000 gaia_source.source_id,gaia_source.ra,gaia_source.dec,gaia_source.parallax,gaia_source.parallax_error,gaia_source.pm,gaia_source.pmra,gaia_source.pmra_error,gaia_source.pmdec,gaia_source.pmdec_error,gaia_source.phot_g_mean_mag,gaia_source.phot_bp_mean_mag,gaia_source.phot_rp_mean_mag,gaia_source.bp_rp,gaia_source.radial_velocity,gaia_source.radial_velocity_error
FROM gaiadr3.gaia_source 
WHERE 
CONTAINS(
	POINT('ICRS',gaiadr3.gaia_source.ra,gaiadr3.gaia_source.dec),
	BOX('ICRS',265.17,-53.68,2,1.5)
)=1
```

# 1. Classification

1. Load the data and cross match the confident cluster members with the larger Gaia sample.

2. Explore the data, sticking to position (e.g., ra, dec, etc.) and velocity (e.g. pm, etc.) measurements for now.  Are there particular observed quantities that seem useful for distinguishing cluster members from background stars?  Be sure to include lots of figures and discussion!

3. Build and train a model (the type of model is up to you!) for classifying stars as members or non-members of NGC 6397 based on Gaia observations.  Be sure to show your process for building and improving the model.

4. How is your final model performing?  Could it have overfit the data?  Is it clear what the model learned?

5. Does your model find any new cluster member candidates?  If so, explore their properties.  Do they seem compelling?  Be sure to connect this discussion back to your response to 1.1.

# 2. Isochrones

Recall that stellar clusters are gravitationally bound groupings of stars born at the same time.  By looking at the H-R diagram of a cluster -- specifically where the population _leaves_ the main sequence -- we can determine the age of the cluster.  To do this we must know the typical main sequence lifetime of stars, which depends on their mass, metalicity, etc. and for that we must turn to stellar modeling.  [MESA](https://docs.mesastar.org) is a stellar modeling library which incorporates the physics important for stellar evolution into 1-D models of stars.  The results useful for our purposes are stellar evolution tracks, the expected trajectory of the star through the H-R diagram as it evolves in time.

[MIST](http://waps.cfa.harvard.edu/MIST/index.html) is an effort to supply a large database of MESA stellar track simulations for the purpose of producing isochrones.  If you imagine stacking up a cluster's worth of stellar evolution tracks, slicing through them at fixed age will produce an isochrone.

I've used the [MIST isochrone interpolator](http://waps.cfa.harvard.edu/MIST/interp_isos.html) to produce an isochrone based on known properties of NGC 6397 (metalicity, reddening due to dust, etc.), and saved `BP-RP`, `Gaia_G_EDR3=phot_g_mean_mag` (with distance modulus already applied), and phase (indicating stellar evolutionary phase) to a CSV file, `data/NGC6397_iso.csv`.

1. Read in and plot the isochrone in all its messy glory.  Try encoding stellar phase information (e.g., using color) if you can, to get a better grasp on the various stellar phases we're looking at (remember, more info on what this indicates can be found [here](http://waps.cfa.harvard.edu/MIST/README_tables.pdf)).

2. Plot all of the Gaia data we pulled along with the isochrone.  In a new figure, plot the cluster members identified in `NGC6397-1.dat` with the isochrone.

3. Use the isochrone to argue whether any new cluster candidates your classifier found are viable.

4. Now include color and brightness information in your model and retrain your classifier.  Does it perform any better?  Did you expect the outcome?

# Graduate Students

1. Use a completely different technique to classify cluster members.  How do your results compare to previous attempts?

2. Do you think your models have learned anything useful for identifying members of other clusters?  Why or why not?

3. Can you think of ways we could use the isochrone explicitly to improve our model?